# Clase 15: SQL básico con Python

**Idea central:** SQL es el lenguaje universal para consultar bases de datos — toda base de datos empresarial se consulta así.

En esta clase aprenderemos a usar SQL desde Python para consultar datos estructurados. Usaremos SQLite, una base de datos que vive completamente en memoria y no necesita instalación de servidor.

## ¿Qué es SQL y por qué lo necesitamos?

SQL (Structured Query Language) es el lenguaje estándar para trabajar con bases de datos relacionales.

**¿Por qué importa para un científico de datos?**

Los datos del mundo real no viven en archivos CSV. Viven en bases de datos:
- Las ventas de una empresa → base de datos MySQL o PostgreSQL
- Los registros médicos de un hospital → SQL Server u Oracle
- Las transacciones de un banco → sistemas propietarios, todos consultables con SQL

SQL es declarativo: describes **qué** quieres, no **cómo** obtenerlo. El motor de base de datos elige el camino más eficiente.

In [ ]:
# Qué hace: crea una base de datos SQLite en memoria y carga el dataset de ventas
# Para qué sirve: preparar el entorno de trabajo para todas las consultas de esta clase

import sqlite3
import pandas as pd

# Cargamos el dataset con pandas como punto de partida
df = pd.read_csv("../../datasets/ventas_tienda.csv")
print("Dataset cargado con pandas:")
print(df.head())
print(f"\nDimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

In [ ]:
# Qué hace: crea la conexión SQLite y vuelca el DataFrame como tabla en la BD
# Para qué sirve: tener los datos disponibles para consultas SQL

import sqlite3
import pandas as pd

df = pd.read_csv("../../datasets/ventas_tienda.csv")

# sqlite3.connect(":memory:") crea una BD en RAM — desaparece al cerrar Python
# Es perfecta para aprender y para análisis en sesión
conn = sqlite3.connect(":memory:")

# to_sql() convierte el DataFrame en una tabla dentro de la BD
# name="ventas" es el nombre que usaremos en las consultas SQL
# if_exists="replace" evita errores si la tabla ya existe
# index=False evita que pandas agregue una columna extra de índice
df.to_sql("ventas", conn, if_exists="replace", index=False)

print("Base de datos en memoria creada.")
print("Tabla disponible: 'ventas'")
print(f"Registros cargados: {len(df)}")

## SELECT — leer datos

La instrucción más fundamental de SQL. Siempre tiene la misma estructura:

```sql
SELECT columnas
FROM tabla
LIMIT cuántas_filas
```

`pd.read_sql(query, conn)` ejecuta la consulta y devuelve un DataFrame con los resultados.

In [ ]:
# Qué hace: ejecuta una consulta SELECT básica sobre la tabla de ventas
# Para qué sirve: verificar que la BD funciona y ver la estructura de los datos

# El asterisco (*) significa "todas las columnas"
# LIMIT 5 devuelve solo las primeras 5 filas — evita traer millones de registros
query = """
    SELECT *
    FROM ventas
    LIMIT 5
"""

# pd.read_sql() ejecuta la consulta y devuelve un DataFrame
resultado = pd.read_sql(query, conn)
print(resultado)

In [ ]:
# Qué hace: selecciona columnas específicas en lugar de todas
# Para qué sirve: traer solo los datos necesarios, lo que es más eficiente en bases de datos grandes

query = """
    SELECT producto, precio_unitario, unidades
    FROM ventas
    LIMIT 10
"""

resultado = pd.read_sql(query, conn)
print(resultado)

# Contar el total de registros en la tabla
query_count = "SELECT COUNT(*) AS total_registros FROM ventas"
print("\nTotal de registros:")
print(pd.read_sql(query_count, conn))

## WHERE — filtrar filas

WHERE aplica una condición y devuelve solo las filas que la cumplen.

Equivalente en pandas: `df[df['columna'] > valor]`

In [ ]:
# Qué hace: filtra filas usando la cláusula WHERE
# Para qué sirve: hacer la misma operación que df[condición] en pandas, pero en SQL

# SQL: ventas con precio mayor a 50
query_sql = """
    SELECT producto, precio_unitario, unidades
    FROM ventas
    WHERE precio_unitario > 50
    ORDER BY precio_unitario DESC
"""
resultado_sql = pd.read_sql(query_sql, conn)
print("SQL — precios > 50:")
print(resultado_sql.head())
print(f"Registros: {len(resultado_sql)}")

# pandas equivalente
resultado_pandas = df[df['precio_unitario'] > 50][['producto', 'precio_unitario', 'unidades']]
resultado_pandas = resultado_pandas.sort_values('precio_unitario', ascending=False)
print("\npandas equivalente:")
print(resultado_pandas.head())
print(f"Registros: {len(resultado_pandas)}")

In [ ]:
# Qué hace: combina condiciones con AND y OR en WHERE
# Para qué sirve: filtros más precisos cuando necesitamos que se cumplan múltiples criterios

# AND: ambas condiciones deben ser verdaderas
query = """
    SELECT producto, precio_unitario, unidades
    FROM ventas
    WHERE precio_unitario > 20
      AND unidades >= 10
"""
resultado = pd.read_sql(query, conn)
print(f"Productos con precio > 20 Y unidades >= 10: {len(resultado)} registros")
print(resultado.head())

# BETWEEN — rango inclusivo
query2 = """
    SELECT producto, precio_unitario
    FROM ventas
    WHERE precio_unitario BETWEEN 15 AND 40
    ORDER BY precio_unitario
"""
resultado2 = pd.read_sql(query2, conn)
print(f"\nPrecios entre 15 y 40: {len(resultado2)} registros")

## GROUP BY — agrupar y resumir

GROUP BY agrupa filas que comparten un valor en una columna y aplica una función de resumen a cada grupo.

Equivalente en pandas: `df.groupby('columna').agg(...)`

Funciones de agregación disponibles: `COUNT`, `SUM`, `AVG`, `MAX`, `MIN`

In [ ]:
# Qué hace: agrupa ventas por categoría y calcula estadísticas de resumen
# Para qué sirve: responder preguntas de negocio como "¿cuánto vende cada categoría?"

# SQL: total de unidades y precio promedio por categoría
query_sql = """
    SELECT
        categoria,
        COUNT(*)             AS cantidad_registros,
        SUM(unidades)        AS total_unidades,
        AVG(precio_unitario) AS precio_promedio,
        MAX(precio_unitario) AS precio_maximo
    FROM ventas
    GROUP BY categoria
    ORDER BY total_unidades DESC
"""
resultado_sql = pd.read_sql(query_sql, conn)
print("SQL — resumen por categoría:")
print(resultado_sql)

# pandas equivalente
resultado_pandas = df.groupby('categoria').agg(
    cantidad_registros=('precio_unitario', 'count'),
    total_unidades=('unidades', 'sum'),
    precio_promedio=('precio_unitario', 'mean'),
    precio_maximo=('precio_unitario', 'max')
).sort_values('total_unidades', ascending=False).reset_index()

print("\npandas equivalente:")
print(resultado_pandas)

## ORDER BY y LIMIT — ordenar y limitar

ORDER BY ordena los resultados. `ASC` es ascendente (por defecto), `DESC` es descendente.

LIMIT restringe cuántas filas devuelve la consulta — muy útil para obtener el Top N.

In [ ]:
# Qué hace: obtiene el top 5 de productos por precio unitario
# Para qué sirve: encontrar valores extremos de forma eficiente

query = """
    SELECT producto, precio_unitario
    FROM ventas
    ORDER BY precio_unitario DESC
    LIMIT 5
"""
top5 = pd.read_sql(query, conn)
print("Top 5 productos más caros:")
print(top5)

# pandas equivalente
print("\npandas equivalente:")
print(df[['producto', 'precio_unitario']].sort_values('precio_unitario', ascending=False).head(5))

## Consulta combinada: análisis completo por categoría

Ahora combinamos todo lo aprendido en una sola consulta que responde múltiples preguntas a la vez.

In [ ]:
# Qué hace: ejecuta una consulta completa con GROUP BY, cálculo de ingreso total y HAVING
# Para qué sirve: demostrar el poder de SQL para responder preguntas complejas en pocas líneas

query = """
    SELECT
        categoria,
        COUNT(*)                            AS registros,
        SUM(unidades)                       AS total_unidades,
        ROUND(AVG(precio_unitario), 2)      AS precio_promedio,
        ROUND(SUM(precio_unitario * unidades), 2) AS ingreso_total
    FROM ventas
    GROUP BY categoria
    HAVING COUNT(*) > 3
    ORDER BY ingreso_total DESC
"""

resultado = pd.read_sql(query, conn)
print("Análisis por categoría (solo categorías con más de 3 registros):")
print(resultado)

print("\nNota: HAVING filtra grupos DESPUÉS de GROUP BY.")
print("WHERE filtra filas ANTES de agrupar — son diferentes y complementarias.")

## Resumen: pandas vs SQL

Ambas herramientas son poderosas. La elección depende del contexto:

| Situación | Herramienta recomendada |
|---|---|
| Los datos viven en una base de datos empresarial | SQL |
| Necesitas manipulación compleja de tablas, merge, pivot | pandas |
| Quieres traer un subconjunto de millones de filas | SQL |
| Vas a graficar o entrenar un modelo | pandas + NumPy |
| El equipo usa un data warehouse (BigQuery, Redshift) | SQL |

En la práctica profesional se usan **juntos**: SQL para traer los datos, pandas para procesarlos y visualizarlos.

In [ ]:
# Qué hace: cierra la conexión a la base de datos al terminar
# Para qué sirve: liberar recursos de memoria — buena práctica en scripts largos

conn.close()
print("Conexión cerrada. La base de datos en memoria fue liberada.")